In [ ]:
from kafka import KafkaConsumer
import psycopg2
import json
from datetime import datetime

# 1. Połączenie z Postgresem
print("Łączę się z bazą danych...")
conn = psycopg2.connect(
    host="postgres",     
    port=5432,
    database="ztm_data",
    user="postgres",
    password="ztm123"
)
cursor = conn.cursor()
print("Połączono z PostgreSQL!")

# 2. Połączenie z Kafką
print("Łączę się z Kafką...")
consumer = KafkaConsumer(
    'ztm_pozycje_gps',
    bootstrap_servers='broker:29092',
    value_deserializer=lambda v: json.loads(v.decode('utf-8')),
    auto_offset_reset='latest',
    group_id='zapis-do-bazy'
)
print("Połączono z Kafką!\n")

# 3. SQL: UPSERT do pojazdy_aktualne
upsert_sql = """
    INSERT INTO pojazdy_aktualne 
        (vehicle_number, lines, brigade, lat, lon, vehicle_type, time_ztm, last_update)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    ON CONFLICT (vehicle_number) DO UPDATE SET
        lines = EXCLUDED.lines,
        brigade = EXCLUDED.brigade,
        lat = EXCLUDED.lat,
        lon = EXCLUDED.lon,
        vehicle_type = EXCLUDED.vehicle_type,
        time_ztm = EXCLUDED.time_ztm,
        last_update = EXCLUDED.last_update;
"""

# 4. SQL: INSERT do pozycje_historia
insert_history_sql = """
    INSERT INTO pozycje_historia 
        (vehicle_number, lines, lat, lon, vehicle_type, time_ztm, ingestion_time)
    VALUES (%s, %s, %s, %s, %s, %s, %s);
"""

print("Czekam na wiadomości i zapisuję do bazy...")

count = 0
try:
    for message in consumer:
        pojazd = message.value
        
        try:
            time_ztm = datetime.strptime(pojazd['Time'], '%Y-%m-%d %H:%M:%S')
            ingestion = datetime.fromisoformat(pojazd['ingestion_time'])
            
            # nadpisuje aktualną pozycję
            cursor.execute(upsert_sql, (
                pojazd['VehicleNumber'],
                pojazd['Lines'],
                pojazd.get('Brigade', ''),
                pojazd['Lat'],
                pojazd['Lon'],
                pojazd['vehicle_type'],
                time_ztm,
                datetime.now()
            ))
            
            # dopisuje do historii
            cursor.execute(insert_history_sql, (
                pojazd['VehicleNumber'],
                pojazd['Lines'],
                pojazd['Lat'],
                pojazd['Lon'],
                pojazd['vehicle_type'],
                time_ztm,
                ingestion
            ))
            
            count += 1
            if count % 100 == 0:
                conn.commit()
                print(f"[{datetime.now().strftime('%H:%M:%S')}] Zapisano {count} pojazdów do bazy")
        
        except Exception as e:
            print(f"Błąd przy zapisie: {e}")
            conn.rollback()

except KeyboardInterrupt:
    print(f"\nZatrzymano. Łącznie zapisano {count} wiadomości.")
finally:
    conn.commit()
    cursor.close()
    conn.close()
    consumer.close()
    print("Zamknięto połączenia.")